In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_DIR = Path("../../data/raw")
pax = pd.read_csv(DATA_DIR / "AA_PAX_FORECASTING.csv")
airports = pd.read_csv(DATA_DIR / "AA_AIRPORT_INFO.csv")
print("pax:", pax.shape, "| airports:", airports.shape)

## 1. Shape and types

In [ ]:
def overview(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(),
        "pct_missing": (df.isna().mean() * 100).round(2),
    })

overview(pax)

In [ ]:
pax.head()

In [ ]:
overview(airports)

In [ ]:
KEY = ["ORG_AIRPRT", "DES_AIRPRT", "DEP_YEAR", "DEP_MONTH"]
print("(org, des, year, month) is unique:", not pax.duplicated(KEY).any())
print("SF_LOAD_TMS distinct values:", pax["SF_LOAD_TMS"].nunique())

| Column | Meaning | Use |
|---|---|---|
| `Unnamed: 0` | saved row index | drop |
| `DEP_MONTH`, `DEP_YEAR` | departure month / year — data is monthly | time |
| `ORG_AIRPRT`, `DES_AIRPRT` | origin / destination airport code | categorical |
| `OD_PAX` | origin–destination passengers (fractional — confirm with team lead) | target |
| `OD_REV_USD` | origin–destination revenue, USD | leakage (section 9) |
| `SF_LOAD_TMS` | load timestamp, a single value | drop |

One row = one route (origin → destination) in one month.

## 2. Prep

In [ ]:
TARGET = "OD_PAX"
ROUTE = ["ORG_AIRPRT", "DES_AIRPRT"]

pax = pax.drop(columns=["Unnamed: 0", "SF_LOAD_TMS"])
pax["date"] = pd.to_datetime(dict(year=pax["DEP_YEAR"], month=pax["DEP_MONTH"], day=1))
pax["t"] = (pax["DEP_YEAR"] - pax["DEP_YEAR"].min()) * 12 + pax["DEP_MONTH"] - 1  # month index


def add_route_lag(df, col, k, name):
    lagged = df[ROUTE + ["t", col]].assign(t=lambda d: d["t"] + k).rename(columns={col: name})
    return df.merge(lagged, on=ROUTE + ["t"], how="left")

## 3. Missingness

In [ ]:
print("NaN cells in pax:", int(pax.isna().sum().sum()))
months_per_route = pax.groupby(ROUTE).size()
print(f"routes present in every month: {(months_per_route == pax['t'].nunique()).mean():.1%}")
pax.groupby("date").size().plot(figsize=(12, 3), title="Routes present per month")
plt.show()

In [ ]:
state_missing = airports["STATE_PROVNC_CD"].isna().groupby(airports["CNTRY_CD"]).mean()
print("% STATE_PROVNC_CD missing - US:", round(state_missing["US"] * 100, 1),
      "| CA:", round(state_missing["CA"] * 100, 1),
      "| other countries:", round(state_missing.drop(["US", "CA"]).mean() * 100, 1))

- No NaN cells; missing data shows up as missing route-months, concentrated in spring–summer 2020
  (most likely no service — confirm with team lead).
- `STATE_PROVNC_CD` is missing only outside the US and Canada → structural. Not used.

## 4. Target distribution and outliers

In [ ]:
y = pax[TARGET]
print(y.describe())
print("skew raw:", round(y.skew(), 2), "| skew log1p:", round(np.log1p(y).skew(), 2))

log_y = np.log1p(y)
q1, q3 = log_y.quantile([0.25, 0.75])
print(f"log-scale IQR outliers: {((log_y < q1 - 1.5 * (q3 - q1)) | (log_y > q3 + 1.5 * (q3 - q1))).mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
y.plot.hist(bins=80, ax=axes[0], title=TARGET)
log_y.plot.hist(bins=80, ax=axes[1], title=f"log1p({TARGET})")
plt.show()

- Positive, fractional, heavily right-skewed → model `log1p(OD_PAX)`.
- Few outliers on the log scale; the extreme values are hub routes, not errors → kept.

## 5. Time axis

In [ ]:
pax.groupby("date")[TARGET].sum().plot(figsize=(12, 3), title=f"Total {TARGET} per month")
plt.show()

by_ym = pax.pivot_table(index="DEP_YEAR", columns="DEP_MONTH", values=TARGET, aggfunc="sum")
by_ym.div(by_ym.mean(axis=1), axis=0).T.plot(figsize=(12, 3), marker="o", title="Seasonal shape by year")
plt.show()

- COVID break: collapse from March 2020, recovery through 2021–2022 → `is_covid_period` flag
  (Mar 2020 – Dec 2021, an assumption).
- Stable seasonality (winter low, summer peak) → `DEP_MONTH` + same-month-last-year lag.
- The last year is partial (runs to July).

## 6. Categorical cardinality

In [ ]:
print("origins:", pax["ORG_AIRPRT"].nunique(), "| destinations:", pax["DES_AIRPRT"].nunique(),
      "| routes:", pax.groupby(ROUTE).ngroups)

## 7. Join airport info (origin and destination)

In [ ]:
ap = airports.assign(
    lat=lambda d: (d["LAT_DEGREE_QTY"] + d["LAT_MINUTE_QTY"] / 60 + d["LAT_SECOND_QTY"] / 3600)
                  * np.where(d["LAT_HEMSPHR_CD"] == "S", -1, 1),
    lon=lambda d: (d["LNGTD_DEGREE_QTY"] + d["LNGTD_MINUTE_QTY"] / 60 + d["LNGTD_SECOND_QTY"] / 3600)
                  * np.where(d["LNGTD_HEMSPHR_CD"] == "W", -1, 1),
).rename(columns={"LNGST_RUNWAY_FT_QTY": "runway_ft", "ELEVATN_FT_QTY": "elevation_ft", "CNTRY_CD": "country"})
ap = ap[["AIRPRT_CD", "country", "lat", "lon", "runway_ft", "elevation_ft"]]

merged = (
    pax.merge(ap.add_prefix("org_"), left_on="ORG_AIRPRT", right_on="org_AIRPRT_CD", how="left", validate="many_to_one")
       .merge(ap.add_prefix("des_"), left_on="DES_AIRPRT", right_on="des_AIRPRT_CD", how="left", validate="many_to_one")
       .drop(columns=["org_AIRPRT_CD", "des_AIRPRT_CD"])
)

lat1, lon1, lat2, lon2 = (np.radians(merged[c]) for c in ["org_lat", "org_lon", "des_lat", "des_lon"])
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
merged["distance_mi"] = 2 * 3958.8 * np.arcsin(np.sqrt(a))  # haversine
merged["is_domestic"] = ((merged["org_country"] == "US") & (merged["des_country"] == "US")).astype(int)
print("rows with no airport match:", merged["org_lat"].isna().sum() + merged["des_lat"].isna().sum())

## 8. Linear vs non-linear

In [ ]:
rel_df = add_route_lag(merged, TARGET, 12, "pax_lag_12")
rel_df["log1p(pax_lag_12)"] = np.log1p(rel_df["pax_lag_12"])
cols = ["pax_lag_12", "log1p(pax_lag_12)", "distance_mi", "org_runway_ft", "des_runway_ft",
        "org_elevation_ft", "des_elevation_ft", "DEP_MONTH", "DEP_YEAR"]
pd.DataFrame({
    "pearson": rel_df[cols].corrwith(rel_df[TARGET]),
    "spearman": rel_df[cols].corrwith(rel_df[TARGET], method="spearman"),
    "pearson_vs_log_target": rel_df[cols].corrwith(np.log1p(rel_df[TARGET])),
}).round(3)

- Same-month-last-year passengers is the strongest signal, close to linear on the log-log scale.
- Distance: Spearman clearly stronger than Pearson → non-linear.
- Month: near-zero correlation despite strong seasonality → non-monotonic.
- Runway length and elevation: weak.

## 9. Leakage

In [ ]:
print("Spearman(OD_PAX, OD_REV_USD), same month:", round(merged[TARGET].corr(merged["OD_REV_USD"], method="spearman"), 3))

| Column | Known at forecast time? | Decision |
|---|---|---|
| `OD_REV_USD` (same month) | No — only known after the month is flown; moves almost one-for-one with passengers | drop; use `fare_lag_12` |
| `pax_lag_12`, `pax_lag_24` | Yes, if the forecast horizon is ≤ 12 months (confirm with team lead) | keep |
| `DEP_YEAR` | Yes, but trees can't extrapolate to an unseen year | drop |
| Airport attributes | Yes — static | keep |

## 10. Feature set

In [ ]:
COVID_START, COVID_END = "2020-03-01", "2021-12-01"
FEATURES = ["ORG_AIRPRT", "DES_AIRPRT", "DEP_MONTH", "distance_mi", "is_domestic", "org_runway_ft",
            "des_runway_ft", "pax_lag_12", "pax_lag_24", "fare_lag_12", "is_covid_period"]


def build_features(df):
    df = df.assign(fare=df["OD_REV_USD"] / df[TARGET])
    df = add_route_lag(df, TARGET, 12, "pax_lag_12")
    df = add_route_lag(df, TARGET, 24, "pax_lag_24")
    df = add_route_lag(df, "fare", 12, "fare_lag_12")
    df["is_covid_period"] = df["date"].between(COVID_START, COVID_END).astype(int)
    X = df[FEATURES].copy()
    X[ROUTE] = X[ROUTE].astype("category")
    return X, np.log1p(df[TARGET])


X, y_log = build_features(merged)
print("X:", X.shape, "| y:", y_log.shape)
overview(X)

| Feature | Decision | Reason |
|---|---|---|
| `pax_lag_12` | engineered | same route, same month last year — route size + seasonality |
| `pax_lag_24` | engineered | two years back — reference that skips a disrupted year |
| `fare_lag_12` | engineered | last year's average fare — price signal without leakage |
| `DEP_MONTH` | kept | seasonality |
| `ORG_AIRPRT`, `DES_AIRPRT` | kept (categorical) | airport effects; categorical instead of one-hot (cardinality) |
| `distance_mi` | engineered | route length; non-linear relationship |
| `is_domestic` | engineered | domestic vs international routes |
| `org_runway_ft`, `des_runway_ft` | kept | rough proxy for airport size |
| `is_covid_period` | engineered | separates the COVID shock from normal demand |
| `OD_REV_USD` (same month) | dropped | leakage |
| `DEP_YEAR` | dropped | trees can't extrapolate to an unseen year |
| `Unnamed: 0`, `SF_LOAD_TMS` | dropped | no information |
| airport name, city, state, world area, elevation, raw lat/lon | dropped | redundant, mostly missing, weak, or replaced by `distance_mi` |

## 11. Model family

**Gradient-boosted trees (XGBoost)** on `log1p(OD_PAX)`, one model across all routes.

- Skewed target → log scale.
- Many airports / routes → native categorical handling instead of a huge one-hot matrix.
- Non-linear (distance) and non-monotonic (month) effects → trees handle them without manual transforms.
- COVID regime break → trees can split on `is_covid_period`.
- Missing lags (route gaps, first years) → XGBoost handles NaN natively.
- Leakage: same-month revenue removed; only past values used.